<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/dynatree-cre/notebooks/dynatree_cre_mvp_colab_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DynaTree-CRE MVP Benchmark Suite (Google Colab GPU)
### Đề tài Luận văn Thạc sĩ TAPTA — Phương pháp: Dynamic Semantic Prompt Trees for Rehearsal-Free CRE

> **Quy tắc hạ tầng bắt buộc (Colab-Only Policy):** Toàn bộ 100% các lượt chạy thực nghiệm dữ liệu thật (smoke runs, ablations, full 8-task FewRel/TACRED, stress tests Exp B/C, và multi-seed) phải thực thi trên **Google Colab GPU (`device: cuda`)**, không chạy thực nghiệm trên máy MacBook local.

**Ràng buộc giao thức (Protocol v1 Track A):**
- **Backbone:** `bert-base-uncased` (100% Frozen, `eval()` mode cố định)
- **Memory:** $M = 0$ (Strictly Rehearsal-Free: zero raw text, zero feature/generative replay)
- **Shots:** Strict Equal 5-shot trên mọi task (kể cả T1)
- **Prompt Token Budget:** $L = 8$ tokens (Residual Addition $P_{\text{root}} + P_v + \Delta P_r$)
- **Inference:** Task-Agnostic CIL (zero Task ID, zero gold relation)

## 1. CUDA / GPU Preflight Check

In [ ]:
import torch
print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
else:
    raise RuntimeError("Khong tim thay GPU CUDA! Vao Runtime -> Change runtime type -> Chon GPU (T4/V100/A100).")
print("=" * 60)

## 2. Clone / Sync Repository `research-code-lab` (`feat/dynatree-cre`) & Setup Data

In [ ]:
import os
REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
BRANCH = "feat/dynatree-cre"
REPO_DIR = "/content/research-code-lab"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scipy scikit-learn tabulate pytest
!python scripts/setup_protocol_v1_data.py

## 3. Run Unit, Research Invariant & Synthetic E2E Tests

In [ ]:
!pytest tests/dynatree_cre -v

## 4. Persistent Checkpoints & Results Backup (Google Drive)

In [ ]:
from pathlib import Path

USE_GDRIVE = True  # Tu dong luu toan bo checkpoint (.pt) va metrics len Google Drive
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    gdrive_runs = Path('/content/drive/MyDrive/TAPTA_DynaTree_CRE/runs/dynatree_cre')
    gdrive_runs.mkdir(parents=True, exist_ok=True)
    local_runs_parent = Path('/content/research-code-lab/runs')
    local_runs_parent.mkdir(parents=True, exist_ok=True)
    local_runs = local_runs_parent / 'dynatree_cre'
    if local_runs.is_symlink():
        local_runs.unlink()
    elif local_runs.exists():
        import shutil
        shutil.copytree(local_runs, gdrive_runs, dirs_exist_ok=True)
        shutil.rmtree(local_runs)
    local_runs.symlink_to(gdrive_runs)
    print(f'Checkpoint & results symlinked -> {gdrive_runs}')

## 5. Part A — FewRel Track A Full 8-Task MVP Suite (Seed 2021)
Chạy toàn bộ `DT-C0` (`t1_centered_cosine` & `raw_cosine` control), `DT-C1`, `DT-C2`, `DT-C3`, `DT-C4` và kiểm chứng checkpoint resume (`after_T4/state.pt` $\rightarrow$ `T8`).

In [ ]:
# 1. DT-C0 Control: t1_centered_cosine vs raw_cosine (8 tasks, full 8,000 test samples)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c0_prototype_control_fewrel_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c0_prototype_control_fewrel_5shot_seed2021.yaml \
    --similarity-space raw_cosine --threshold-mode fixed --insertion-threshold 0.75 \
    --output-dir runs/dynatree_cre/dt_c0_raw_cosine/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

# 2. DT-C1 .. DT-C4 (8 tasks, full 8,000 test samples)
fewrel_configs = [
    "configs/dynatree_cre/dt_c1_flat_hard_fewrel_5shot_seed2021.yaml",
    "configs/dynatree_cre/dt_c2_flat_soft_fewrel_5shot_seed2021.yaml",
    "configs/dynatree_cre/dt_c3_cluster_prompt_flat_route_fewrel_5shot_seed2021.yaml",
    "configs/dynatree_cre/dt_c4_dynatree_hier_soft_fewrel_5shot_seed2021.yaml",
]
for cfg in fewrel_configs:
    print("=== Running:", cfg)
    !python experiments/run_dynatree_cre.py --config {cfg} --device cuda --allow-hub-download

# 3. Checkpoint Resume Verification on CUDA (after_T4 -> T8)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c4_dynatree_hier_soft_fewrel_5shot_seed2021.yaml \
    --resume-from runs/dynatree_cre/dt_c4_hier_soft/fewrel_5shot/seed_2021/checkpoints/after_T4/state.pt \
    --output-dir runs/dynatree_cre/dt_c4_resume_check/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

## 6. Part B — TACRED Track A Full 8-Task (40 Relations, Seed 2021)

In [ ]:
# DT-C0 Controls on TACRED (t1_centered_cosine & raw_cosine)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c0_prototype_control_tacred_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c0_prototype_control_tacred_5shot_seed2021.yaml \
    --similarity-space raw_cosine --threshold-mode fixed --insertion-threshold 0.75 \
    --output-dir runs/dynatree_cre/dt_c0_raw_cosine/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download

# DT-C4 Main Method on TACRED (40 positive relations)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/dt_c4_dynatree_hier_soft_tacred_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

## 7. Part C — Task Distribution Stress Tests (Experiment B & Experiment C)
Kiểm chứng khả năng chia sẻ tri thức xuyên tác vụ (`DT-C2` vs `DT-C4`) dưới hai phân phối khắc nghiệt:
- **Experiment B (`semantic_homogeneous`):** Các quan hệ cùng miền tập trung theo task.
- **Experiment C (`cross_task_dispersed`):** Các quan hệ cùng miền bị phân tán qua nhiều task.

In [ ]:
for dist in ["semantic_homogeneous", "cross_task_dispersed"]:
    !python experiments/run_dynatree_cre.py \
        --config configs/dynatree_cre/dt_c2_flat_soft_fewrel_5shot_seed2021.yaml \
        --task-distribution {dist} \
        --output-dir runs/dynatree_cre/dt_c2_{dist}/fewrel_5shot/seed_2021 \
        --device cuda --allow-hub-download
    !python experiments/run_dynatree_cre.py \
        --config configs/dynatree_cre/dt_c4_dynatree_hier_soft_fewrel_5shot_seed2021.yaml \
        --task-distribution {dist} \
        --output-dir runs/dynatree_cre/dt_c4_{dist}/fewrel_5shot/seed_2021 \
        --device cuda --allow-hub-download

## 8. Part D — Summary Table & Export Artifacts Bundle (`dynatree_cre_mvp_colab_results.zip`)

In [ ]:
import json
import os
import shutil
from pathlib import Path
from tabulate import tabulate
from google.colab import files

runs_root = Path("/content/research-code-lab/runs/dynatree_cre")
rows = []
for m_path in sorted(runs_root.glob("**/metrics.json")):
    d = json.loads(m_path.read_text())
    r_path = m_path.parent / "routing_metrics.json"
    res_path = m_path.parent / "resource_metrics.json"
    r = json.loads(r_path.read_text()) if r_path.exists() else {}
    res = json.loads(res_path.read_text()) if res_path.exists() else {}
    eval_r = r.get("stages", [{}])[-1].get("eval_routing", {})
    mem_kb = res.get("memory_bytes", {}).get("total_persistent_state_bytes", 0) / 1024
    rows.append([
        m_path.relative_to(runs_root).parts[0],
        d.get("dataset"),
        d.get("task_distribution"),
        f"{d.get('final_cosine_whole_accuracy', 0)*100:.2f}%",
        f"{d.get('final_ncm_whole_accuracy', 0)*100:.2f}%",
        f"{d.get('cosine_summary', {}).get('average_forgetting', 0)*100:.2f}%",
        f"{d.get('ncm_summary', {}).get('average_forgetting', 0)*100:.2f}%",
        d.get("final_tree_metrics", {}).get("num_clusters"),
        f"{eval_r.get('relation_routing_recall_at_k', 0)*100:.2f}%" if eval_r else "N/A",
        f"{mem_kb:.1f} KB",
    ])

headers = ["Run", "Dataset", "Dist", "Cos Acc", "NCM Acc", "Cos F", "NCM F", "Clusters", "Recall@K", "Mem"]
print(tabulate(rows, headers=headers, tablefmt="grid"))

export_dir = Path("/content/dynatree_cre_mvp_bundle")
if export_dir.exists():
    shutil.rmtree(export_dir)
shutil.copytree(runs_root, export_dir, ignore=shutil.ignore_patterns("*.pt"))
zip_path = "/content/dynatree_cre_mvp_colab_results"
shutil.make_archive(zip_path, "zip", export_dir)
print(f"Exported: {zip_path}.zip ({os.path.getsize(zip_path + '.zip') / 1024:.1f} KB)")
files.download(zip_path + ".zip")